# 🛢️ Oil Spill Detection — Module 1 Training

## ⚠️ BEFORE YOU RUN
> 1. **Enable GPU**: Settings (⚙️) → Accelerator → **GPU T4 x1** → Save
> 2. **Add Kaggle Secret**: Account → Settings → Secrets → `HF_TOKEN`
>    (Get this from huggingface.co → Settings → Access Tokens)
> 3. Set `HF_REPO_ID` at the top of **Cell 1** (e.g. `"username/oil-spill-checkpoints"`)

## 📋 How This Works (Save & Run Mode)
- Cell 1 tests Hugging Face connection **before** starting the 12-hour run — fails fast if broken
- Cell 4 training script **auto-uploads** `best_model.pt` to Hugging Face after every val-loss improvement
- Also uploads `last_model.pt` + `train_metrics.csv` every 5 epochs
- **Account 2 resume**: run Cells 1 → 3 → 4 — Hugging Face checkpoint is downloaded automatically

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — GPU VERIFICATION  (run first, always)
# ═══════════════════════════════════════════════════════════════════════════════
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Fix: Settings (⚙️) → Accelerator → GPU T4 x1 → Save → Factory Reset session."
    )

device_name = torch.cuda.get_device_name(0)
total_vram  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ GPU : {device_name} ({total_vram:.1f} GB VRAM)")
print(f"   CUDA: {torch.version.cuda}   PyTorch: {torch.__version__}")
print("\n🟢 GPU ready — proceed to Cell 1.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP: Repo · Dependencies · Hugging Face Connection Test
#
# ─── CONFIGURE HERE ──────────────────────────────────────────────────────────
# Your Hugging Face repository ID (where checkpoints will be saved)
# Example: "rohithsheregar/oil-spill-checkpoints"
HF_REPO_ID = ""
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json

# ── 1a. Clone / pull latest repo ─────────────────────────────────────────────
REPO_URL = "https://github.com/Rohith-Sheregar/Oil-Spill-Detection-New.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Repo: {os.getcwd()}")

# ── 1b. Install dependencies ──────────────────────────────────────────────────
print("\n📦 Installing deps...")
!pip install -q segmentation-models-pytorch albumentations scikit-image \
              scipy joblib imagecodecs huggingface_hub
print("✅ Dependencies ready.")

# ── 1c. Load HF Token from Kaggle Secret ─────────────────────────────────────
HF_TOKEN = ""
print(f"\n🔑 Loading Hugging Face token from Kaggle Secret 'HF_TOKEN'...")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Token loaded successfully.")
except Exception as _e:
    print(f"⚠️  Could not load secret 'HF_TOKEN': {_e}")
    print("   Uploads will be DISABLED. Training will still run locally.")

# ── 1d. Hugging Face connection test ─────────────────────────────────────────
print("\n🧪 Testing Hugging Face connection...")

if not HF_REPO_ID:
    print("⚠️  HF_REPO_ID is empty — skipping HF test.")
    print("   Set HF_REPO_ID at the top of this cell to enable auto-save.")
elif not HF_TOKEN:
    print("⚠️  No HF_TOKEN found — skipping HF test.")
else:
    try:
        import time
        from huggingface_hub import HfApi
        
        _api = HfApi(token=HF_TOKEN)
        
        # Verify token and create repo if it doesn't exist
        _user = _api.whoami()['name']
        print(f"   ✅ Authenticated as: {_user}")
        
        _api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
        print(f"   ✅ Repo ready: https://huggingface.co/datasets/{HF_REPO_ID}")
        
        # Write a 2-byte test file and upload it
        _test_name = f"_kaggle_hf_test_{int(time.time())}.txt"
        _test_path = f"/kaggle/working/{_test_name}"
        with open(_test_path, "w") as _tf:
            _tf.write("ok")

        _api.upload_file(
            path_or_fileobj=_test_path,
            path_in_repo=_test_name,
            repo_id=HF_REPO_ID,
            commit_message="Kaggle Connection Test"
        )
        print(f"   ✅ Upload OK  → {_test_name}")

        # Cleanup
        _api.delete_file(path_in_repo=_test_name, repo_id=HF_REPO_ID)
        os.remove(_test_path)
        print(f"   ✅ Cleanup OK → test file deleted")
        print()
        print("🟢 Hugging Face Hub connected! Auto-save is ACTIVE.")

    except Exception as _e:
        print(f"❌ Hugging Face test FAILED: {_e}")
        print()
        print("🔧 Common fixes:")
        print("   • Token missing 'write' permission? (Generate a new token with write access)")
        print("   • Wrong repo ID format? (Must be 'username/repo-name')")
        print("   • Secret not enabled? (Kaggle sidebar 🔒 → toggle HF_TOKEN ON)")
        raise  # Stop here — don't waste 12 hrs with broken HF connection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA SYMLINKS + SANITY CHECK
# ═══════════════════════════════════════════════════════════════════════════════
import glob, os, shutil
from pathlib import Path

INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"📂 Input root: {INPUT_DIR}")
available_dirs = os.listdir(INPUT_DIR)
for d in available_dirs:
    print(f"   └─ {d}")

working_data_dir = "/kaggle/working/data"
if os.path.exists(working_data_dir):
    shutil.rmtree(working_data_dir)

mappings = {
    "train/oil":       lambda n: "oil" in n and not any(k in n for k in ["lookalike", "no", "test"]),
    "train/lookalike": lambda n: "lookalike" in n,
    "train/no_oil":    lambda n: "no" in n and "oil" in n,
    "test/oil":        lambda n: "test" in n,
}

total_tiffs = 0
print("\n🔗 Creating symlinks...")
for subpath, cond in mappings.items():
    matched = [d for d in available_dirs if cond(d.lower())]
    if not matched:
        print(f"   ❌ WARNING: no dataset matched for '{subpath}'")
        continue
    src = os.path.join(INPUT_DIR, matched[0])
    dst = os.path.join(working_data_dir, subpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(src, "**", "*.tif*"), recursive=True))
    total_tiffs += n
    print(f"   ✅ {subpath} → {matched[0]}  ({n} TIFFs)")

print(f"\n📊 Total TIFFs: {total_tiffs}")

print("\n🔍 Dataset sanity check...")
from src.training.zenodo_sos_dataset import discover_sos_pairs
df_oil = discover_sos_pairs(Path("/kaggle/working/data/train/oil"), include_classes=["oil"])
print(f"✅ {len(df_oil)} oil scene pairs found.")
if len(df_oil) == 0:
    raise RuntimeError("❌ No oil pairs found! Check Kaggle dataset attachments.")
r = df_oil.iloc[0]
print(f"   Scene: {r['scene_id']}  |  image: {r['image_path']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — RESUME CHECKPOINT DETECTION
#
# Priority 1: Download directly from Hugging Face Hub (works cross-account!)
# Priority 2: Fresh start
#
# For Account 2 resuming from Account 1:
#   → Just set HF_REPO_ID in Cell 1 (same repo Account 1 wrote to)
#   → Run this cell — it downloads last_model.pt directly from HF
# ─────────────────────────────────────────────────────────────────────────────
import os, glob

PREFER_CHECKPOINT   = "last_model.pt"   # or "best_model.pt"
DOWNLOAD_DIR = "/kaggle/working/resume_ckpt"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

RESUME_CKPT = None

# ── Priority 1: Download from Hugging Face ───────────────────────────────────
if HF_REPO_ID and HF_TOKEN:
    print(f"🔍 Checking Hugging Face ({HF_REPO_ID}) for checkpoints...")
    try:
        from huggingface_hub import HfFileSystem, hf_hub_download
        
        _fs = HfFileSystem(token=HF_TOKEN)
        _files = _fs.ls(HF_REPO_ID, detail=False)
        
        _checkpoints = [f['name'].split('/')[-1] for f in _files if f['name'].endswith('.pt')]
        
        if _checkpoints:
            print(f"   Found {len(_checkpoints)} checkpoint(s) in HF Hub:")
            for _f in _checkpoints:
                print(f"     • {_f}")

            _target = PREFER_CHECKPOINT if PREFER_CHECKPOINT in _checkpoints else _checkpoints[-1]
            
            print(f"\n   ⬇️  Downloading '{_target}' from HF Hub...")
            _dest = hf_hub_download(
                repo_id=HF_REPO_ID, 
                filename=_target, 
                token=HF_TOKEN,
                local_dir=DOWNLOAD_DIR
            )
            print(f"   ✅ Downloaded to: {_dest}")
            RESUME_CKPT = _dest
        else:
            print("   ℹ️  No .pt files in HF Hub yet — fresh start.")
    except Exception as _e:
        print(f"   ⚠️  Hugging Face download failed: {_e}")

# ── Result ───────────────────────────────────────────────────────────────────
if RESUME_CKPT:
    import torch
    _ckpt  = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
    _epoch = _ckpt.get("epoch", "?")
    _loss  = _ckpt.get("val_loss", "?")
    _miou  = _ckpt.get("val_miou", "?")
    print(f"\n▶  RESUMING FROM : {RESUME_CKPT}")
    print(f"   Saved epoch    : {_epoch}")
    print(f"   Best val_loss  : {_loss}")
    print(f"   Best mIoU      : {_miou}")
    print(f"   ➡️  Will train from epoch {int(_epoch)+1}")
else:
    print("\n🆕 No checkpoint found — fresh start.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — TRAINING  (auto-uploads to Hugging Face after every best epoch)
# ═══════════════════════════════════════════════════════════════════════════════
import time, torch

# ─── SESSION CONFIG ──────────────────────────────────────────────────────────
# Epochs to train THIS session.
# T4 GPU: ~20 epochs safely fits in 12 hrs.
# P100  : ~30 epochs.
EPOCHS_THIS_SESSION = 20

# Pseudo-label cycles:
# 0 = skip (use on all intermediate sessions)
# 5 = run  (FINAL session only, after all epochs done)
PSEUDO_CYCLES = 0
# ─────────────────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Run Cell 0 first.")

print(f"⚡ GPU      : {torch.cuda.get_device_name(0)}")
print(f"📅 Epochs   : {EPOCHS_THIS_SESSION}")
print(f"▶  Resume   : {RESUME_CKPT or 'None (fresh start)'}")
print(f"☁️  HF Hub   : {'ACTIVE → ' + HF_REPO_ID if HF_REPO_ID else 'DISABLED'}")
print()

resume_flag      = f"--resume {RESUME_CKPT}"          if RESUME_CKPT         else ""
pseudo_flag      = f"--pseudo-cycles {PSEUDO_CYCLES}" if PSEUDO_CYCLES > 0   else "--no-pseudo"
skip_pseudo_flag = "--skip-pseudo-on-resume"          if RESUME_CKPT         else ""
hf_flags         = (
    f"--hf-repo-id {HF_REPO_ID} --hf-token {HF_TOKEN}"
    if HF_REPO_ID and HF_TOKEN else ""
)

start = time.time()

!python -m src.training.train_module1 \
    --data-root /kaggle/working/data \
    --results-dir /kaggle/working/results/module1 \
    --input-mode full_5band \
    --epochs {EPOCHS_THIS_SESSION} \
    --lr 1e-3 \
    --batch-size 16 \
    --num-workers 2 \
    {resume_flag} \
    {pseudo_flag} \
    {skip_pseudo_flag} \
    {hf_flags}

print(f"\n🏁 Done in {(time.time()-start)/3600:.2f} hrs")
print(f"📁 Local : /kaggle/working/results/module1/checkpoints/")
if HF_REPO_ID:
    print(f"☁️  HF Hub : https://huggingface.co/datasets/{HF_REPO_ID}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — SESSION SUMMARY + LOCAL OUTPUT COLLECTION
# Run at end of session to see metrics and collect files for the Output tab.
# (Checkpoints are already in HF Hub if auto-save was active.)
# ═══════════════════════════════════════════════════════════════════════════════
import os, shutil, glob, csv
from pathlib import Path

CHECKPOINT_SRC = "/kaggle/working/results/module1/checkpoints"
METRICS_SRC    = "/kaggle/working/results/module1/metrics"
OUT            = "/kaggle/working/session_output"
os.makedirs(OUT, exist_ok=True)

print("📦 Collecting files for Output tab...")
for pat in [f"{CHECKPOINT_SRC}/*.pt", f"{METRICS_SRC}/*.csv", f"{METRICS_SRC}/*.json"]:
    for src in glob.glob(pat):
        dst     = shutil.copy(src, OUT)
        size_mb = os.path.getsize(dst) / (1024**2)
        print(f"   ✅ {Path(src).name}  ({size_mb:.1f} MB)")

print(f"\n📂 Files at: {OUT}")

# Training summary
csv_p = f"{METRICS_SRC}/train_metrics.csv"
if os.path.exists(csv_p):
    with open(csv_p) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = rows[-1]
        best = min(rows, key=lambda r: float(r['val_loss']))
        print("\n📊 Training Summary")
        print(f"   {'Epochs logged':<22}: {len(rows)}")
        print(f"   {'Last epoch':<22}: {last['epoch']}")
        print(f"   {'Last val_loss':<22}: {float(last['val_loss']):.4f}")
        print(f"   {'Last mIoU':<22}: {float(last['val_miou']):.4f}")
        print(f"   {'Best val_loss':<22}: {float(best['val_loss']):.4f}  @ epoch {best['epoch']}")
        print(f"   {'Best mIoU':<22}: {float(best['val_miou']):.4f}  @ epoch {best['epoch']}")

if HF_REPO_ID:
    print(f"\n☁️  Checkpoints also auto-saved to Hugging Face Hub:")
    print(f"   https://huggingface.co/datasets/{HF_REPO_ID}")

---
## 🔄 Account 2 Resume Guide

### What you need on Account 2
- Same Kaggle Secret: `HF_TOKEN` (add via Account → Settings → Secrets)
- Same `HF_REPO_ID` pasted in Cell 1
- Same Kaggle datasets attached (SOS training data)

### Cells to run on Account 2 (in order)

| Cell | Run | Notes |
|------|-----|-------|
| Cell 0 | ✅ | GPU check |
| Cell 1 | ✅ | Same `HF_REPO_ID` → HF test should pass |
| Cell 2 | ✅ | Data symlinks — attach same datasets |
| Cell 3 | ✅ | Auto-downloads `last_model.pt` from HF Hub — **no manual transfer needed** |
| Cell 4 | ✅ | Set `EPOCHS_THIS_SESSION`, training resumes from saved epoch |
| Cell 5 | optional | Summary |

### What you do NOT need
- A Kaggle Dataset with checkpoints
- Manual file download / upload between accounts
- Any `KAGGLE_CKPT_DATASET` setting

> Cell 3 downloads the checkpoint directly from Hugging Face using the token. Cell 4 resumes training from that exact epoch and continues auto-uploading back to the same repo.